In [ ]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
def timeseries_of_pot_center_variable_hourly (pot_center_variable_path, pot_time_col,
                        original_dataset_center_variable_path, original_dataset_center_variable_time_col, original_dataset_center_variable_col_name,
                        original_dataset_associated_variable_1_path, original_dataset_associated_variable_1_time_col, original_dataset_associated_variable_1_col_name,
                        lag_hour_col_variable_1,
                        original_dataset_associated_variable_2_path, original_dataset_associated_variable_2_time_col, original_dataset_associated_variable_2_col_name,
                        lag_hour_col_variable_2,
                        output_path
                        ):

    df_pot_center_variable = pd.read_csv(pot_center_variable_path)

    df_timeseries_ntr = pd.read_csv(original_dataset_center_variable_path)
    df_timeseries_ntr = df_timeseries_ntr.rename(columns={original_dataset_center_variable_time_col:'time'})
    df_timeseries_ntr = df_timeseries_ntr[['time', original_dataset_center_variable_col_name]]
    
    df_timeseries_rf = pd.read_csv(original_dataset_associated_variable_1_path)
    df_timeseries_rf = df_timeseries_rf.rename(columns={original_dataset_associated_variable_1_time_col:'time'})
    df_timeseries_rf = df_timeseries_rf[['time', original_dataset_associated_variable_1_col_name]]
    
    df_timeseries_rd = pd.read_csv(original_dataset_associated_variable_2_path)
    df_timeseries_rd = df_timeseries_rd.rename(columns={original_dataset_associated_variable_2_time_col:'time'})
    df_timeseries_rd = df_timeseries_rd[['time', original_dataset_associated_variable_2_col_name]]
    
    df_timeseries_rd['time'] = pd.to_datetime(
        df_timeseries_rd['time'].astype(str).str.strip(),
        format='%m/%d/%Y',   
        errors='coerce'
    )
    df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.strftime('%Y-%m-%d %H:%M:%S')     
    
    df_pot_center_variable['time']          = pd.to_datetime(df_pot_center_variable[pot_time_col], errors='coerce')
    df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr['time'], errors='coerce')
    df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf['time'],  errors='coerce')
    df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd['time'],  errors='coerce')
    
    # --- Build an HOURLY base from RF ⨉ NTR ---
    hourly_base = (
        df_timeseries_rf[['time', original_dataset_center_variable_col_name]]
            .merge(df_timeseries_ntr[['time', original_dataset_associated_variable_1_col_name]], 
                   on='time', how='inner')  
            .sort_values('time')
    )
    hourly_base['date'] = hourly_base['time'].dt.normalize() 
    
    # --- Map the DAILY RD value to each hour by DATE  ---
    df_rd_daily = df_timeseries_rd[['time', original_dataset_associated_variable_2_col_name]].copy()
    df_rd_daily['date'] = df_rd_daily['time'].dt.normalize()
    df_rd_daily = df_rd_daily[['date', original_dataset_associated_variable_2_col_name]]
    hourly_base = hourly_base.merge(df_rd_daily, on='date', how='left')
    
    # ---------------------------
    # Windows around each POT NTR
    # ---------------------------
    center_variable_windows = []
    
    for _, row in df_pot_center_variable.iterrows():
        pot_time = row['time']
        window_start = pot_time - pd.Timedelta(days=5)
        window_end   = pot_time + pd.Timedelta(days=5)
    
        # hourly window (RF+NTR, with RD value repeated for each hour of the day)
        window_df = hourly_base.loc[
            hourly_base['time'].between(window_start, window_end)
        ].copy()
        window_df['event_time'] = pot_time
    
        # --- max NTR within ±5 days using HOURLY NTR ---
        sub_ntr = window_df.loc[
            window_df['time'].between(pot_time - pd.Timedelta(days=5),
                                      pot_time + pd.Timedelta(days=5)),
            ['time', original_dataset_associated_variable_1_path]
        ].dropna()
    
        if not sub_ntr.empty:
            i = sub_ntr[original_dataset_associated_variable_1_path].idxmax()
            max_ntr_time = sub_ntr.at[i, 'time']
            window_df[lag_hour_col_variable_1] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
        else:
            window_df[lag_hour_col_variable_1] = np.nan
    
        # --- max RD within ±5 days using the ORIGINAL DAILY RD series ---
        sub_rd = df_timeseries_rd.loc[
            df_timeseries_rd['time'].between(pot_time - pd.Timedelta(days=5),
                                             pot_time + pd.Timedelta(days=5)),
            ['time', original_dataset_associated_variable_2_path]
        ].dropna()
    
        if not sub_rd.empty:
            j = sub_rd[original_dataset_associated_variable_2_path].idxmax()
            max_rd_time = sub_rd.at[j, 'time']        # daily timestamp (likely 00:00:00)
            window_df[lag_hour_col_variable_2] = (max_rd_time - pot_time) / pd.Timedelta(hours=1)
        else:
            window_df[lag_hour_col_variable_2] = np.nan
    
        center_variable_windows.append(window_df)
    
    df_windows_total = pd.concat(center_variable_windows, ignore_index=True)
    df_windows_total = df_windows_total.sort_values(['event_time', 'time']).reset_index(drop=True)
    
    # Keep datetime dtype; only format on CSV write if needed:
    df_windows_total.to_csv(output_path, index=False, date_format='%Y-%m-%d %H:%M:%S')
    return df_windows_total

In [ ]:
def expand_daily_to_hourly(df_daily, time_col='time', value_col=original_dataset_center_variable_col_name):
        
    """
    Expands daily time-series data into hourly resolution by repeating each
    daily value for all 24 hours of the corresponding day.

    Parameters
    ----------
    df_daily : pandas.DataFrame
        DataFrame containing daily observations.

    time_col : str
        Column name representing the date/time (daily resolution).

    value_col : str
        Column name of the variable to be expanded (e.g., river discharge).

    Returns
    -------
    pandas.DataFrame
        Hourly time-series DataFrame where each daily value is repeated
        for 24 hourly timestamps (00:00–23:00).

    Notes
    -----
    - The function assumes that each daily value applies uniformly across
      the entire day.
    - The resulting time series will have one row per hour.
    - Missing or non-numeric values are removed before expansion.

    Physical Interpretation
    -----------------------
    Daily variables such as river discharge are assumed constant within each day,
    allowing them to be aligned with hourly variables (e.g., NTR, RF).
    """
    
    df_daily = df_daily.dropna(subset=[time_col, value_col]).copy()
    df_daily[time_col] = pd.to_datetime(df_daily[time_col], errors='coerce')
    df_daily[value_col] = pd.to_numeric(df_daily[value_col], errors='coerce')
    df_daily = df_daily.dropna(subset=[time_col, value_col])

    pieces = []
    for t, v in zip(df_daily[time_col], df_daily[value_col]):
        d = pd.Timestamp(t).normalize()  # midnight of that day
        hours = pd.date_range(d, d + pd.Timedelta(hours=23), freq='H')
        pieces.append(pd.DataFrame({time_col: hours, value_col: v}))
    if not pieces:
        return df_daily.iloc[0:0][[time_col, value_col]]  # empty with same cols
    return pd.concat(pieces, ignore_index=True)
    

In [ ]:
def timeseries_of_pot_center_variable_daily (pot_center_variable_path, pot_time_col,
                        original_dataset_center_variable_path, original_dataset_center_variable_time_col, original_dataset_center_variable_col_name,
                        original_dataset_associated_variable_1_path, original_dataset_associated_variable_1_time_col, original_dataset_associated_variable_1_col_name,
                        lag_hour_col_variable_1,
                        original_dataset_associated_variable_2_path, original_dataset_associated_variable_2_time_col, original_dataset_associated_variable_2_col_name,
                        lag_hour_col_variable_2,
                        output_path
                        ):
    df_pot_center_variable = pd.read_csv(pot_center_variable_path)

    df_timeseries_rd = pd.read_csv(original_dataset_center_variable_path)
    df_timeseries_rd = df_timeseries_ntr.rename(columns={original_dataset_center_variable_time_col:'time'})
    df_timeseries_rd = df_timeseries_ntr[['time', original_dataset_center_variable_col_name]]
    
    df_timeseries_rf = pd.read_csv(original_dataset_associated_variable_1_path)
    df_timeseries_rf = df_timeseries_rf.rename(columns={original_dataset_associated_variable_1_time_col:'time'})
    df_timeseries_rf = df_timeseries_rf[['time', original_dataset_associated_variable_1_col_name]]
    
    df_timeseries_ntr = pd.read_csv(original_dataset_associated_variable_2_path)
    df_timeseries_ntr = df_timeseries_rd.rename(columns={original_dataset_associated_variable_2_time_col:'time'})
    df_timeseries_ntr = df_timeseries_rd[['time', original_dataset_associated_variable_2_col_name]]
    
    df_timeseries_rd['time'] = pd.to_datetime(
        df_timeseries_rd['time'].astype(str).str.strip(),
        format='%m/%d/%Y',   
        errors='coerce'
    )
    df_timeseries_rd['time'] = df_timeseries_rd['time'].dt.strftime('%Y-%m-%d %H:%M:%S')     
    
    df_pot_center_variable['time']          = pd.to_datetime(df_pot_center_variable[pot_time_col], errors='coerce')
    df_timeseries_ntr['time']  = pd.to_datetime(df_timeseries_ntr['time'], errors='coerce')
    df_timeseries_rf['time']   = pd.to_datetime(df_timeseries_rf['time'],  errors='coerce')
    df_timeseries_rd['time']   = pd.to_datetime(df_timeseries_rd['time'],  errors='coerce')

    center_variable_windows = [] 
    for _, row in df_pot_center_variable.iterrows():
        pot_time = row['time']
        window_start = pot_time - pd.Timedelta(days=5)
        window_end   = pot_time + pd.Timedelta(days=5)
    
        window_df_rd_daily = df_timeseries_rd.loc[
            (df_timeseries_rd['time'] >= window_start) &
            (df_timeseries_rd['time'] <= window_end)
        ].copy()
    
        # ⬇️ expand daily RD to hourly within the window
        window_df_rd = expand_daily_to_hourly(
            window_df_rd_daily, time_col='time', value_col=original_dataset_center_variable_col_name
        )
    
        # keep NTR/RF hourly (unchanged)
        window_df_ntr = df_timeseries_ntr.loc[
            (df_timeseries_ntr['time'] >= window_start) &
            (df_timeseries_ntr['time'] <= window_end)
        ].copy()
    
        window_df_rf = df_timeseries_rf.loc[
            (df_timeseries_rf['time'] >= window_start) &
            (df_timeseries_rf['time'] <= window_end)
        ].copy()
    
        # inner-join on hourly timestamps; RD is now hourly by repetition
        window_df = (
            window_df_rd
            .merge(window_df_ntr, on='time', how='inner')
            .merge(window_df_rf, on='time', how='inner')
            .sort_values('time')
            .reset_index(drop=True)
        )
        window_df['event_time'] = pot_time
    
        if not window_df.empty:
            window_start = pot_time - pd.Timedelta(days=5)
            window_end   = pot_time + pd.Timedelta(days=5)
            sub = window_df.loc[
            window_df['time'].between(window_start, window_end),
                ['time', original_dataset_associated_variable_1_col_name]
            ].dropna(subset=[original_dataset_associated_variable_1_col_name])
            
            if not sub.empty:
                i = sub[original_dataset_associated_variable_1_col_name].idxmax()
                max_rf_time = sub.at[i, 'time']
                max_rf_val  = sub.at[i, original_dataset_associated_variable_1_col_name]
                window_df[lag_hour_col_variable_1] = (max_rf_time - pot_time) / pd.Timedelta(hours=1)
        
            sub = window_df.loc[
            window_df['time'].between(window_start, window_end),
                ['time', original_dataset_associated_variable_2_col_name]
            ].dropna(subset=[original_dataset_associated_variable_2_col_name])
            
            if not sub.empty:
                i = sub[ntr_col].idxmax()
                max_ntr_time = sub.at[i, 'time']
                max_ntr_val  = sub.at[i, ntr_col]
                window_df[lag_hour_col_variable_2] = (max_ntr_time - pot_time) / pd.Timedelta(hours=1)
                
            else:
                window_df[lag_hour_col_variable_1] = np.nan
                window_df[lag_hour_col_variable_2]  = np.nan
    
        center_variable_windows.append(window_df)
    
    df_windows_total = pd.concat(center_variable_windows, ignore_index=True)

    df_windows_total.to_csv(output_path, index=False)
    return df_windows_total
